# E16_2 — E15 final.pt 진단 (재학습 없음)
E16과 동일한 정책·보상·종료 조건 및 프레임 진단을 사용합니다. 공개 트랙 1000–1009를 진단하고 기존 E16/E15의 best 결과와 비교합니다. 방향 판정은 근사이므로 영상과 대조하세요.

위에서부터 전체 실행하세요. E15 실행 때 RUN_NAME을 바꿨다면 아래 E15_DIR을 수정하세요. 결과는 `MyDrive/261103/261103_results/E16_2/<실행시각>/`에 저장됩니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil, tempfile
ROOT=Path('/content/drive/MyDrive/261103')
RESULTS=ROOT/'261103_results'
if not (ROOT/'train.py').exists():
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(['git','clone','--depth','1','--branch','261103','https://github.com/MilkySon/RL_carracing_competition.git',tmp+'/repo'],check=True)
        shutil.copytree(Path(tmp)/'repo'/'261103',ROOT,dirs_exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
os.environ['SDL_VIDEODRIVER']='dummy'
os.environ['SDL_AUDIODRIVER']='dummy'
print(ROOT,RESULTS)


Mounted at /content/drive
/content/drive/MyDrive/261103 /content/drive/MyDrive/261103/261103_results


In [2]:
subprocess.run([sys.executable,'-m','pip','install','-q','swig'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('GPU:',torch.cuda.is_available())
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a') as f:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='');f.write(line);f.flush()
        rc=p.wait()
    if rc:raise RuntimeError(f'Failed ({rc}): {log}')


GPU: True


## 1. final 모델과 출력 경로
final.pt가 없으면 중단합니다. best.pt로 자동 대체하지 않습니다. 기존 E16 결과는 읽기만 합니다.

In [3]:
from datetime import datetime, timezone
import hashlib, platform
E15_DIR=RESULTS/'E15'
FINAL_CKPT=E15_DIR/'final.pt'
BEST_EPISODES=RESULTS/'E16'/'E15'/'episodes.csv'
SEEDS='1000-1009'
VIDEO_SEEDS='1000,1001,1005,1007'
if not FINAL_CKPT.is_file():
    raise FileNotFoundError(f'E15 final.pt가 없습니다. E15_DIR을 확인하세요: {FINAL_CKPT}')
CHECKPOINTS={'E15_final':FINAL_CKPT}
OUT=RESULTS/'E16_2'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT.mkdir(parents=True,exist_ok=False)
manifest={'checkpoint':str(FINAL_CKPT),'checkpoint_sha256':hashlib.sha256(FINAL_CKPT.read_bytes()).hexdigest(),
    'seeds':SEEDS,'video_seeds':VIDEO_SEEDS,'python':platform.python_version(),'torch':torch.__version__,
    'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'source_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in ROOT.rglob('*.py') if '261103_results' not in p.parts}}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('진단 모델:',FINAL_CKPT)
print('결과 경로:',OUT)


진단 모델: /content/drive/MyDrive/261103/261103_results/E15/final.pt
결과 경로: /content/drive/MyDrive/261103/261103_results/E16_2/20261104T023231_510306Z


## 2. 종료 조건 및 프레임 로그
이탈=모든 바퀴의 도로 접촉 없음. 도로 재접촉과 정방향·새 타일 방문 복귀를 구별합니다. 연속 역주행 집계는 원본 프레임 단위입니다.

In [4]:
for name,ckpt in CHECKPOINTS.items():
    if not ckpt.exists():print('SKIP missing:',name,ckpt);continue
    run([sys.executable,'audit_termination.py','--ckpt',ckpt,'--seeds','1000-1009','--out',OUT/name],OUT/f'{name}_console.log')
import pandas as pd
for p in sorted(OUT.glob('*/episodes.csv')):
    print(p.parent.name);display(pd.read_csv(p))


/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
{'seed': 1000, 'return_': 835.1535836177296, 'raw_frames': 1000, 'offroad_frames': 4, 'exits': 2, 'road_reentries': 2, 'forward_progress_recoveries': 2, 'reverse_frames': 0, 'max_reverse_streak': 0, 'terminated': False, 'truncated': True, 'end_reason': 'time_limit'}
{'seed': 1001, 'return_': 765.384615384603, 'raw_frames': 1000, 'offroad_frames': 2, 'exits': 1, 'road_reentries': 1, 'forward_progress_recoveries': 1, 'reverse_frames': 0, 'max_reverse_streak': 0, 'terminated': False, 'truncated': True, 'end_reason': 'time_limit'}
{'seed': 1002, 'return_': 841.8181818181663, 'raw_frames': 1000, 'offroad_frames': 10, 'exits': 1, 'ro

,seed,return_,raw_frames,offroad_frames,exits,road_reentries,forward_progress_recoveries,reverse_frames,max_reverse_streak,terminated,truncated,end_reason
0,1000,835.153584,1000,4,2,2,2,0,0,False,True,time_limit
1,1001,765.384615,1000,2,1,1,1,0,0,False,True,time_limit
2,1002,841.818182,1000,10,1,1,1,0,0,False,True,time_limit
3,1003,736.666667,1000,2,1,1,1,0,0,False,True,time_limit
4,1004,772.483221,1000,2,1,1,1,0,0,False,True,time_limit
5,1005,734.355828,1000,0,0,0,0,0,0,False,True,time_limit
6,1006,775.000000,1000,3,1,1,1,0,0,False,True,time_limit
7,1007,741.423948,1000,2,1,1,1,0,0,False,True,time_limit
8,1008,684.810127,1000,8,3,3,3,0,0,False,True,time_limit
9,1009,877.777778,1000,0,0,0,0,0,0,False,True,time_limit


## 3. 주요 트랙 영상
기존 best의 실패 트랙 1000·1005·1007과 비교용 1001을 저장합니다. 진단과 영상은 별도 rollout입니다.

In [5]:
for name,ckpt in CHECKPOINTS.items():
    run([sys.executable,'evaluate.py','--ckpt',ckpt,'--seeds',VIDEO_SEEDS,'--video',OUT/name/'videos'],OUT/f'{name}_video_console.log')
from IPython.display import Video, display
videos=sorted((OUT/'E15_final'/'videos').glob('*.mp4'))
if videos:display(Video(str(videos[0]),embed=True,width=400))


/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
seed 1000:    835.2
seed 1001:    765.4
seed 1005:    734.4
seed 1007:    741.4
SCORE (mean over 4 tracks): 769.1 ± 39.8


## 4. 기존 best와 비교
점수 차이는 양수, 이탈·역주행 프레임 차이는 음수일 때 개선입니다. 700점은 큰 실패 추적용 임시 기준입니다.

In [6]:
import numpy as np
final=pd.read_csv(OUT/'E15_final'/'episodes.csv')
def summarize(df,label):
    scores=df['return_'].to_numpy()
    k=max(1,int(np.ceil(len(scores)*0.2)))
    return dict(model=label,tracks=len(scores),mean=float(scores.mean()),std=float(scores.std(ddof=0)),
        below700=int((scores<700).sum()),bottom20_mean=float(np.sort(scores)[:k].mean()),
        offroad_frames=int(df.offroad_frames.sum()),reverse_frames=int(df.reverse_frames.sum()),
        max_reverse_streak=int(df.max_reverse_streak.max()))
rows=[summarize(final,'E15_final')]
if BEST_EPISODES.is_file():
    best=pd.read_csv(BEST_EPISODES)
    if best.seed.duplicated().any() or final.seed.duplicated().any():
        raise ValueError('중복 seed가 있습니다. 기존 episodes.csv를 확인하세요.')
    if set(best.seed)!=set(final.seed):
        print('주의: seed 집합이 다릅니다. 트랙별 비교는 공통 seed만 사용합니다.')
    cols=['seed','return_','offroad_frames','reverse_frames','max_reverse_streak','forward_progress_recoveries']
    comparison=best[cols].merge(final[cols],on='seed',suffixes=('_best','_final'),validate='one_to_one')
    for metric in cols[1:]:
        comparison[metric+'_delta_final_minus_best']=comparison[metric+'_final']-comparison[metric+'_best']
    comparison.to_csv(OUT/'best_vs_final_by_seed.csv',index=False)
    display(comparison)
    common=comparison.seed
    rows=[summarize(best[best.seed.isin(common)],'E15_best_common'),
          summarize(final[final.seed.isin(common)],'E15_final_common')] if len(common) else rows
    print('기존 best 결과와의 비교입니다. 장치·코드·패키지 차이는 각 audit.json과 manifest로 확인하세요.')
else:
    print('기존 best 진단이 없어 비교만 생략합니다:',BEST_EPISODES)
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'summary.csv',index=False)
display(summary)
print('완료:',OUT)


,seed,return__best,offroad_frames_best,reverse_frames_best,max_reverse_streak_best,forward_progress_recoveries_best,return__final,offroad_frames_final,reverse_frames_final,max_reverse_streak_final,forward_progress_recoveries_final,return__delta_final_minus_best,offroad_frames_delta_final_minus_best,reverse_frames_delta_final_minus_best,max_reverse_streak_delta_final_minus_best,forward_progress_recoveries_delta_final_minus_best
0,1000,572.354949,131,723,723,0,835.153584,4,0,0,2,262.798635,-127,-723,-723,2
1,1001,598.717949,81,214,175,4,765.384615,2,0,0,1,166.666667,-79,-214,-175,-3
2,1002,863.636364,72,91,91,3,841.818182,10,0,0,1,-21.818182,-62,-91,-91,-2
3,1003,783.333333,52,32,32,3,736.666667,2,0,0,1,-46.666667,-50,-32,-32,-2
4,1004,886.577181,3,0,0,1,772.483221,2,0,0,1,-114.093960,-1,0,0,0
5,1005,34.844172,81,33,33,0,734.355828,0,0,0,0,699.511656,-81,-33,-33,0
6,1006,915.700000,0,0,0,0,775.000000,3,0,0,1,-140.700000,3,0,0,1
7,1007,272.168285,425,261,182,0,741.423948,2,0,0,1,469.255663,-423,-261,-182,1
8,1008,687.974684,166,37,37,2,684.810127,8,0,0,3,-3.164557,-158,-37,-37,1
9,1009,918.600000,0,0,0,0,877.777778,0,0,0,0,-40.822222,0,0,0,0


기존 best 결과와의 비교입니다. 장치·코드·패키지 차이는 각 audit.json과 manifest로 확인하세요.


,model,tracks,mean,std,below700,bottom20_mean,offroad_frames,reverse_frames,max_reverse_streak
0,E15_best_common,10,653.390692,281.653812,5,153.506228,1011,1391,723
1,E15_final_common,10,776.487395,55.772770,1,709.582977,33,0,0


완료: /content/drive/MyDrive/261103/261103_results/E16_2/20261104T023231_510306Z


## 판단 메모
frames.csv: 이탈 → 재접촉 → 정방향/새 타일 진행의 순서 확인. 종료 원인은 관측에 따른 분류이며 영상·원본 환경 코드와 대조.
즉시 종료 규칙이 없으므로 잔디 20스텝 허용을 무조건 추가하지 않습니다. 복귀 실패가 반복될 때 E17, 이후 필요 시 E18 검토.